In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
TOROOTPATH   = "../../../model/"
INPUT_PATH   = TOROOTPATH + "data/raw/factory_sensor_simulator_2040.csv"
OUT_DIR      = TOROOTPATH + "data/process/factory_sensor/"
OUT_X_TRAIN  = OUT_DIR + "X_train.csv"
OUT_X_TEST   = OUT_DIR + "X_test.csv"
OUT_Y_TRAIN  = OUT_DIR + "y_train.csv"
OUT_Y_TEST   = OUT_DIR + "y_test.csv"
 
TARGET       = "Remaining_Useful_Life_days"
TEST_SIZE    = 0.2
RANDOM_STATE = 42
CURRENT_YEAR = 2040          # context year for age calculation
 
WINSOR_LOW   = 0.01          # 1st  percentile
WINSOR_HIGH  = 0.99          # 99th percentile
 
# rough domain maxima used to scale the Stress_Index feature
# (named so they are no longer "magic numbers" buried in the formula)
TEMP_NORM    = 100.0         # approx. max Temperature_C
VIB_NORM     = 25.0          # approx. max Vibration_mms
 
# sensors winsorised (tail-clipped) using TRAIN-only quantiles
SENSOR_COLS = [
    "Temperature_C", "Vibration_mms", "Sound_dB",
    "Oil_Level_pct", "Coolant_Level_pct", "Power_Consumption_kW",
]
 
# physically non-negative quantities -> negatives are sensor errors
NONNEG_COLS = ["Vibration_mms", "Power_Consumption_kW", "Sound_dB"]
 
# percentages must sit within [0, 100]
PCT_COLS = ["Oil_Level_pct", "Coolant_Level_pct"]
 
# machine-specific columns: NaN means "sensor not applicable to this machine"
MACHINE_SPECIFIC_COLS = [
    "Laser_Intensity",
    "Hydraulic_Pressure_bar",
    "Coolant_Flow_L_min",
    "Heat_Index",
]
 
# columns to drop: identifiers + target leakage.
# Installation_Year is consumed by add_machine_age() and then dropped here.
DROP_COLS = ["Machine_ID", "Failure_Within_7_Days", "Installation_Year"]
 
 
def section(title: str) -> None:
    print(f"\n{'─' * 60}")
    print(f"  {title}")
    print(f"{'─' * 60}")
 

C:\Users\Admin\AppData\Local\Temp\ipykernel_9012\550070744.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# ──────────────────────────────────────────────────────────────
# STEP 1 — LOAD
# ──────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    section("STEP 1 — Load")
    df = pd.read_csv(path)
    print(f"  Raw shape: {df.shape}")
    return df

In [3]:
# ──────────────────────────────────────────────────────────────
# STEP 2 — VALIDATE SCHEMA (fail fast on missing columns)
# ──────────────────────────────────────────────────────────────
def validate_schema(df: pd.DataFrame) -> None:
    section("STEP 2 — Validate Schema")
    required = set(
        [TARGET, "Machine_Type", "AI_Supervision", "Operational_Hours",
         "Maintenance_History_Count", "Failure_History_Count",
         "Error_Codes_Last_30_Days", "Last_Maintenance_Days_Ago"]
        + SENSOR_COLS
    )
    missing = sorted(required - set(df.columns))
    if missing:
        raise KeyError(f"Input is missing required columns: {missing}")
    print(f"  OK — all {len(required)} required columns present")

In [4]:
# ──────────────────────────────────────────────────────────────
# STEP 3 — DERIVE Machine_Age  (BEFORE Installation_Year is dropped)
# ──────────────────────────────────────────────────────────────
def add_machine_age(df: pd.DataFrame) -> pd.DataFrame:
    """
    The original code referenced a non-existent 'Installation_Year_orig'
    column, so Machine_Age silently became a constant 0. Here we compute it
    correctly from Installation_Year while it still exists; if the column is
    absent we skip the feature instead of fabricating zeros.
    """
    section("STEP 3 — Derive Machine_Age")
    if "Installation_Year" in df.columns:
        df["Machine_Age"] = (CURRENT_YEAR - df["Installation_Year"]).clip(lower=0)
        n_nan = int(df["Machine_Age"].isna().sum())
        print(f"  + Machine_Age = {CURRENT_YEAR} - Installation_Year"
              f"  (NaN: {n_nan:,} -> imputed later)")
    else:
        print("  Installation_Year not found — skipping Machine_Age")
    return df

In [5]:
# ──────────────────────────────────────────────────────────────
# STEP 4 — DROP IDENTIFIER / LEAKAGE COLUMNS
# ──────────────────────────────────────────────────────────────
def drop_columns(df: pd.DataFrame) -> pd.DataFrame:
    section("STEP 4 — Drop Identifier / Leakage Columns")
    present = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=present)
    print(f"  Dropped: {present}")
    print("  reason: Machine_ID = high-cardinality id, "
          "Failure_Within_7_Days = target leakage")
    print(f"  Remaining columns: {df.shape[1]}")
    return df

In [6]:
# ──────────────────────────────────────────────────────────────
# STEP 5 — DATA QUALITY FIXES
# ──────────────────────────────────────────────────────────────
def quality_fixes(df: pd.DataFrame) -> pd.DataFrame:
    section("STEP 5 — Data Quality Fixes")
 
    # 5a. exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    print(f"  [5a] Duplicate rows removed: {before - len(df):,}")
    #      NOTE: dedup runs after Machine_ID was dropped, so two *different*
    #      machines with identical readings would be merged. With continuous
    #      sensor floats this is effectively impossible; if your data is
    #      panel/time-series, dedup on a logical key BEFORE dropping the id.
 
    # 5b. physically-impossible values -> NaN (imputed later on TRAIN medians).
    #     Using NaN+impute instead of abs(): abs(-5) -> +5 invents a wrong
    #     magnitude, whereas NaN says "this reading is unusable".
    for col in NONNEG_COLS:
        if col in df.columns:
            mask = df[col] < 0
            n = int(mask.sum())
            if n:
                df.loc[mask, col] = np.nan
                print(f"  [5b] {col}: {n:,} negative -> NaN")
 
    for col in PCT_COLS:
        if col in df.columns:
            mask = (df[col] < 0) | (df[col] > 100)
            n = int(mask.sum())
            if n:
                df.loc[mask, col] = np.nan
                print(f"  [5b] {col}: {n:,} outside [0,100] -> NaN")
    #      Temperature_C is intentionally NOT forced non-negative — sub-zero
    #      Celsius can be legitimate; its extreme tails are handled by winsorising.
 
    # 5c. a null TARGET is unusable -> drop only those rows
    #     (feature NaNs are imputed later instead of dropping whole rows)
    before = len(df)
    df = df.dropna(subset=[TARGET])
    print(f"  [5c] Rows dropped (null target): {before - len(df):,}")
 
    print(f"  Shape after quality fixes: {df.shape}")
    return df

In [7]:
# ──────────────────────────────────────────────────────────────
# STEP 6 — ENCODE CATEGORICALS
# ──────────────────────────────────────────────────────────────
def encode_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    section("STEP 6 — Encode Categoricals")
 
    # Machine_Type -> one-hot. Done before the split so train/test share the
    # same columns. No drop_first: XGBoost is a tree model and is unaffected
    # by the dummy collinearity that drop_first guards against.
    dummies = pd.get_dummies(df["Machine_Type"], prefix="mtype", dtype=int)
    df = pd.concat([df.drop(columns=["Machine_Type"]), dummies], axis=1)
    print(f"  Machine_Type -> {dummies.shape[1]} one-hot columns")
 
    # AI_Supervision -> 0/1, robust to bool / numeric / 'Yes'-'No' strings / NaN.
    # (The original .astype(int) crashes on NaN or strings. We also check
    #  is_numeric_dtype rather than == object, because modern pandas reads text
    #  columns as the 'string' dtype, which == object would silently miss,
    #  turning every "Yes"/"No" into NaN -> 0.)
    ai = df["AI_Supervision"]
    if not pd.api.types.is_numeric_dtype(ai):
        bool_map = {"yes": 1, "no": 0, "true": 1, "false": 0,
                    "y": 1, "n": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}
        ai = ai.astype(str).str.strip().str.lower().map(bool_map)
    df["AI_Supervision"] = pd.to_numeric(ai, errors="coerce").fillna(0).astype(int)
    print("  AI_Supervision -> int (0/1)")
    return df

In [8]:
# ──────────────────────────────────────────────────────────────
# STEP 7 — FLAG + FILL MACHINE-SPECIFIC SENSORS
# ──────────────────────────────────────────────────────────────
def flag_and_fill_machine_specific(df: pd.DataFrame) -> pd.DataFrame:
    section("STEP 7 — Flag + Fill Machine-Specific Sensors")
    # Add a binary presence flag, then fill NaN with 0 (= sensor not applicable).
    # The flag lets XGBoost separate "not applicable" from a genuine 0 reading.
    for col in MACHINE_SPECIFIC_COLS:
        if col not in df.columns:
            print(f"  {col}: not found — skipped")
            continue
        flag = f"{col}_present"
        df[flag] = df[col].notna().astype(int)
        df[col] = df[col].fillna(0)
        print(f"  {col}: + '{flag}', NaN -> 0")
    return df

In [9]:
# ──────────────────────────────────────────────────────────────
# STEP 8 — TRAIN / TEST SPLIT  (everything after this is fit on TRAIN only)
# ──────────────────────────────────────────────────────────────
def split_data(df: pd.DataFrame):
    section("STEP 8 — Train / Test Split")
    X = df.drop(columns=[TARGET])
    y = df[TARGET]
 
    mtype_cols = [c for c in X.columns if c.startswith("mtype_")]
    strat = X[mtype_cols].idxmax(axis=1) if mtype_cols else None
 
    if strat is not None and strat.value_counts().min() < 2:
        print("  (a machine type has <2 rows — disabling stratify)")
        strat = None
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat,
    )
    print(f"  Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")
    print(f"  Features: {X_train.shape[1]}")
    print(f"  Target mean — train: {y_train.mean():.1f}  test: {y_test.mean():.1f}")
    print(f"  Target std  — train: {y_train.std():.1f}   test: {y_test.std():.1f}")
    return X_train, X_test, y_train, y_test

In [10]:
# ──────────────────────────────────────────────────────────────
# STEP 9 — IMPUTE REMAINING NaN  (medians fit on TRAIN, applied to both)
# ──────────────────────────────────────────────────────────────
def impute_missing(X_train: pd.DataFrame, X_test: pd.DataFrame):
    section("STEP 9 — Impute Remaining NaN (train medians)")
    medians = X_train.median(numeric_only=True)          # fit on TRAIN only
    reported = sorted(
        set(X_train.columns[X_train.isna().any()])
        | set(X_test.columns[X_test.isna().any()])
    )
    X_train = X_train.fillna(medians)
    X_test = X_test.fillna(medians)                       # TRAIN medians on TEST
    if reported:
        for c in reported:
            if c in medians.index:
                print(f"  {c}: filled with train median = {medians[c]:.3f}")
    else:
        print("  No remaining NaN to impute")
    return X_train, X_test
 
 

In [11]:
# ──────────────────────────────────────────────────────────────
# STEP 10 — WINSORISE  (bounds fit on TRAIN, applied to both) — NO LEAKAGE
# ──────────────────────────────────────────────────────────────
def winsorise(X_train: pd.DataFrame, X_test: pd.DataFrame):
    section("STEP 10 — Winsorise Sensor Outliers (train bounds)")
    for col in SENSOR_COLS:
        if col not in X_train.columns:
            continue
        lo = X_train[col].quantile(WINSOR_LOW)            # TRAIN quantiles only
        hi = X_train[col].quantile(WINSOR_HIGH)
        n_tr = int(((X_train[col] < lo) | (X_train[col] > hi)).sum())
        X_train[col] = X_train[col].clip(lo, hi)
        X_test[col] = X_test[col].clip(lo, hi)           
        print(f"  {col}: clip [{lo:.2f}, {hi:.2f}]  (train rows affected: {n_tr:,})")
    return X_train, X_test

In [12]:
# ──────────────────────────────────────────────────────────────
# STEP 11 — FEATURE ENGINEERING  (row-wise -> safe per split, no leakage)
# ──────────────────────────────────────────────────────────────
def engineer_features(X: pd.DataFrame) -> pd.DataFrame:
    """
    All features here are row-wise (no global statistics), so applying the
    same function independently to train and test introduces no leakage.
    Runs AFTER winsorising, so Stress_Index / Error_Power_Interaction use the
    clipped sensor values (the original built them from raw, pre-clip data).
    """
    X = X.copy()
 
    X["Hours_Per_Maintenance"] = np.where(
        X["Maintenance_History_Count"] > 0,
        X["Operational_Hours"] / X["Maintenance_History_Count"],
        X["Operational_Hours"],                     
    )
 
    X["Failure_Rate_10k"] = (
        X["Failure_History_Count"] / (X["Operational_Hours"] + 1)
    ) * 10_000
 
    X["Stress_Index"] = (X["Temperature_C"] / TEMP_NORM) + (X["Vibration_mms"] / VIB_NORM)
 
    X["Maintenance_Overdue"] = (X["Last_Maintenance_Days_Ago"] > 365).astype(int)
 
    X["Error_Power_Interaction"] = (
        X["Error_Codes_Last_30_Days"] * X["Power_Consumption_kW"]
    )
    return X

In [13]:
# ──────────────────────────────────────────────────────────────
# STEP 12 — FINAL NaN AUDIT
# ──────────────────────────────────────────────────────────────
def nan_audit(X_train: pd.DataFrame, X_test: pd.DataFrame) -> None:
    section("STEP 12 — Final NaN Audit")
    tr = int(X_train.isna().sum().sum())
    te = int(X_test.isna().sum().sum())
    print(f"  NaN in X_train: {tr}")
    print(f"  NaN in X_test : {te}")
    if tr or te:
        bad = (X_train.isna().sum() + X_test.isna().sum())
        bad = bad[bad > 0]
        raise ValueError(f"Unexpected NaN remaining:\n{bad}")
    print("  ✓ no NaN — ready for XGBoost")

In [14]:
# ──────────────────────────────────────────────────────────────
# STEP 13 — SAVE
# ──────────────────────────────────────────────────────────────
def save_outputs(X_train, X_test, y_train, y_test) -> None:
    section("STEP 13 — Save")
    os.makedirs(OUT_DIR, exist_ok=True)
    X_train.to_csv(OUT_X_TRAIN, index=False)
    X_test.to_csv(OUT_X_TEST, index=False)
    y_train.to_csv(OUT_Y_TRAIN, index=False, header=True)
    y_test.to_csv(OUT_Y_TEST, index=False, header=True)
    for p in (OUT_X_TRAIN, OUT_X_TEST, OUT_Y_TRAIN, OUT_Y_TEST):
        print(f"  ✓ {p}")

In [15]:
# ──────────────────────────────────────────────────────────────
# STEP 14 — FEATURE SUMMARY
# ──────────────────────────────────────────────────────────────
def feature_summary(X_train: pd.DataFrame, y_train: pd.Series) -> None:
    section("STEP 14 — Feature Summary")
    print(f"\n  Total features -> XGBoost: {X_train.shape[1]}\n")

    machine_specific = []
    for col in MACHINE_SPECIFIC_COLS:
        if col in X_train.columns:
            machine_specific.append(col)
        flag = f"{col}_present"
        if flag in X_train.columns:
            machine_specific.append(flag)
 
    groups = {
        "Sensor readings (winsorised)": SENSOR_COLS,
        "Operational history": [
            "Operational_Hours", "Last_Maintenance_Days_Ago",
            "Maintenance_History_Count", "Failure_History_Count",
            "Error_Codes_Last_30_Days", "AI_Override_Events",
        ],
        "Machine-specific sensors (+ presence flags)": machine_specific,
        "Engineered features": [
            "Machine_Age", "Hours_Per_Maintenance", "Failure_Rate_10k",
            "Stress_Index", "Maintenance_Overdue", "Error_Power_Interaction",
        ],
        "One-hot: Machine_Type": [c for c in X_train.columns if c.startswith("mtype_")],
        "Flags": ["AI_Supervision"],
    }
 
    seen = set()
    for name, cols in groups.items():
        present = [c for c in cols if c in X_train.columns]
        seen.update(present)
        print(f"  [{len(present):2d}] {name}")
        for c in present:
            # guard against constant columns (corr -> NaN + a divide warning)
            if X_train[c].nunique() < 2:
                print(f"        • {c:<34s} |corr|=  n/a (constant)")
            else:
                corr = abs(X_train[c].corr(y_train))
                print(f"        • {c:<34s} |corr|={corr:.4f}")
 
    leftover = [c for c in X_train.columns if c not in seen]
    if leftover:
        print(f"  [{len(leftover):2d}] Uncategorised")
        for c in leftover:
            print(f"        • {c}")
 
    total_grouped = len(seen) + len(leftover)
    print(f"\n  (groups + leftover = {total_grouped} = {X_train.shape[1]} total)\n")
    print("  NOTE: 'Failure_Within_7_Days' EXCLUDED — target leakage.")
    print("  NOTE: 'Machine_ID' EXCLUDED — high-cardinality id, no signal.")
    print("  Ready for:  xgb.XGBRegressor().fit(X_train, y_train)")

In [16]:
# ──────────────────────────────────────────────────────────────
# ORCHESTRATION
# ──────────────────────────────────────────────────────────────
def main() -> None:
    df = load_data(INPUT_PATH)
    validate_schema(df)
    df = add_machine_age(df)        # before Installation_Year is dropped
    df = drop_columns(df)
    df = quality_fixes(df)
    df = encode_categoricals(df)
    df = flag_and_fill_machine_specific(df)
 
    # ---- split FIRST, then fit every statistic on TRAIN only ----
    X_train, X_test, y_train, y_test = split_data(df)
    X_train, X_test = impute_missing(X_train, X_test)
    X_train, X_test = winsorise(X_train, X_test)
 
    section("STEP 11 — Feature Engineering")
    X_train = engineer_features(X_train)
    X_test = engineer_features(X_test)
    print("  + Hours_Per_Maintenance, Failure_Rate_10k, Stress_Index,")
    print("    Maintenance_Overdue, Error_Power_Interaction")
 
    nan_audit(X_train, X_test)
    save_outputs(X_train, X_test, y_train, y_test)
    feature_summary(X_train, y_train)
    
     
if __name__ == "__main__":
    main()


────────────────────────────────────────────────────────────
  STEP 1 — Load
────────────────────────────────────────────────────────────
  Raw shape: (500000, 22)

────────────────────────────────────────────────────────────
  STEP 2 — Validate Schema
────────────────────────────────────────────────────────────
  OK — all 14 required columns present

────────────────────────────────────────────────────────────
  STEP 3 — Derive Machine_Age
────────────────────────────────────────────────────────────
  + Machine_Age = 2040 - Installation_Year  (NaN: 0 -> imputed later)

────────────────────────────────────────────────────────────
  STEP 4 — Drop Identifier / Leakage Columns
────────────────────────────────────────────────────────────
  Dropped: ['Machine_ID', 'Failure_Within_7_Days', 'Installation_Year']
  reason: Machine_ID = high-cardinality id, Failure_Within_7_Days = target leakage
  Remaining columns: 20

────────────────────────────────────────────────────────────
  STEP 5 — Dat